In [55]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from math import radians
import matplotlib.pyplot as plt

In [56]:
# Load all 13 stakeholder datasets
def load_datasets():
    """Load all 13 stakeholder datasets with proper encoding"""
    print("Loading datasets...")
    
    datasets = {
        "clubs": pd.read_csv("data/Clubs_Dataset_with_id.csv", encoding="ISO-8859-1"),
        "suppliers": pd.read_csv("data/Equipment_suppliers_updated.csv", encoding="ISO-8859-1"),
        "fitness_clubs": pd.read_csv("data/Fitness_clubs.csv", encoding="ISO-8859-1"),
        "traveling_agencies": pd.read_csv("data/Traveling_agencies_Dataset.csv", encoding="ISO-8859-1"),
        "service_providers": pd.read_csv("data/Service_Providers_Dataset.csv", encoding="ISO-8859-1"),
        "sponsors": pd.read_csv("data/Sponsors_Dataset.csv", encoding="ISO-8859-1"),
        "sporting_management_agencies": pd.read_csv("data/Sporting_management_agencies_Dataset.csv", encoding="ISO-8859-1"),
        "communication_boxes": pd.read_csv("data/Communication_Boxes_Dataset.csv", encoding="ISO-8859-1"),
        "players_agents": pd.read_csv("data/Players_Agents_Dataset.csv", encoding="ISO-8859-1"),
        "recruiting_agents": pd.read_csv("data/Recruiting_Agents_Dataset.csv", encoding="ISO-8859-1"),
        "managers_staff": pd.read_csv("data/Managers_Staff_Dataset.csv", encoding="ISO-8859-1"),
        "sports_clothing_brands": pd.read_csv("data/Sports_Clothing_Brands_Dataset.csv", encoding="ISO-8859-1"),
        "players": pd.read_csv("data/Players_Dataset.csv", encoding="ISO-8859-1")
    }
    
    for name, df in datasets.items():
        print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")
        
    return datasets

# Load datasets
datasets = load_datasets()

Loading datasets...
clubs: 511 rows, 19 columns
suppliers: 1911 rows, 9 columns
fitness_clubs: 288 rows, 10 columns
traveling_agencies: 590 rows, 9 columns
service_providers: 404 rows, 8 columns
sponsors: 228 rows, 11 columns
sporting_management_agencies: 586 rows, 7 columns
communication_boxes: 628 rows, 9 columns
players_agents: 500 rows, 19 columns
recruiting_agents: 245 rows, 11 columns
managers_staff: 500 rows, 13 columns
sports_clothing_brands: 600 rows, 9 columns
players: 948 rows, 49 columns


In [57]:
# Define field mappings for location, ID, and name fields
def get_field_mappings():
    """Get mappings for location fields, ID fields, and name fields"""
    
    # Location field mappings
    location_fields = {
        'players': 'nationality',
        'clubs': 'country',
        'players_agents': 'region',
        'recruiting_agents': 'region',
        'service_providers': 'region',
        'sporting_management_agencies': 'region',
        'communication_boxes': 'region',
        'fitness_clubs': 'location',
        'suppliers': 'location',
        'sports_clothing_brands': 'location',
        'traveling_agencies': 'operating_regions',
        'sponsors': 'location',
        'managers_staff': None  # No clear location field
    }
    
    # ID field mappings
    id_fields = {
        'players': 'player_id',
        'clubs': 'id',
        'managers_staff': 'person',  # No clear ID, using name
        'players_agents': 'license_number',
        'recruiting_agents': 'scout_id',
        'service_providers': 'provider_id',
        'sporting_management_agencies': 'agency_id',
        'communication_boxes': 'box_id',
        'fitness_clubs': 'club_id',
        'suppliers': 'supplier_id',
        'sports_clothing_brands': 'brand_id',
        'traveling_agencies': 'agency_id',
        'sponsors': 'sponsor_id'
    }
    
    # Name field mappings
    name_fields = {
        'players': 'full_name',
        'clubs': 'club name',
        'managers_staff': 'person',
        'players_agents': 'full_name',
        'recruiting_agents': 'full_name',
        'service_providers': 'company_name',
        'sporting_management_agencies': 'agency_name',
        'communication_boxes': 'company_name',
        'fitness_clubs': 'club_name',
        'suppliers': 'company_name',
        'sports_clothing_brands': 'brand_name',
        'traveling_agencies': 'agency_name',
        'sponsors': 'company_name'
    }
    
    return location_fields, id_fields, name_fields

# Get field mappings
location_fields, id_fields, name_fields = get_field_mappings()

In [58]:
# Define coordinates for African locations
def get_africa_location_coordinates():
    """
    Get coordinates for African countries, regions, and major cities
    """
    # Countries
    country_coords = {
        'Algeria': (28.0339, 1.6596),
        'Angola': (-11.2027, 17.8739),
        'Benin': (9.3077, 2.3158),
        'Botswana': (-22.3285, 24.6849),
        'Burkina Faso': (12.2383, -1.5616),
        'Burundi': (-3.3731, 29.9189),
        'Cameroon': (7.3697, 12.3547),
        'Cape Verde': (16.5388, -23.0418),
        'Central African Republic': (6.6111, 20.9394),
        'Chad': (15.4542, 18.7322),
        'Comoros': (-11.6455, 43.3333),
        'Congo': (-0.2280, 15.8277),
        'Democratic Republic of Congo': (-4.0383, 21.7587),
        'Djibouti': (11.8251, 42.5903),
        'Egypt': (26.8206, 30.8025),
        'Equatorial Guinea': (1.6508, 10.2679),
        'Eritrea': (15.1794, 39.7823),
        'Eswatini': (-26.5225, 31.4659),
        'Ethiopia': (9.1450, 40.4897),
        'Gabon': (-0.8037, 11.6094),
        'Gambia': (13.4432, -15.3101),
        'Ghana': (7.9465, -1.0232),
        'Guinea': (9.9456, -9.6966),
        'Guinea-Bissau': (11.8037, -15.1804),
        'Ivory Coast': (7.5400, -5.5471),
        'Kenya': (0.0236, 37.9062),
        'Lesotho': (-29.6100, 28.2336),
        'Liberia': (6.4281, -9.4295),
        'Libya': (26.3351, 17.2283),
        'Madagascar': (-18.7669, 46.8691),
        'Malawi': (-13.2543, 34.3015),
        'Mali': (17.5707, -3.9962),
        'Mauritania': (21.0079, -10.9408),
        'Mauritius': (-20.3484, 57.5522),
        'Morocco': (31.7917, -7.0926),
        'Mozambique': (-18.6657, 35.5296),
        'Namibia': (-22.9576, 18.4904),
        'Niger': (17.6078, 8.0817),
        'Nigeria': (9.0820, 8.6753),
        'Rwanda': (-1.9403, 29.8739),
        'Senegal': (14.4974, -14.4524),
        'Seychelles': (-4.6796, 55.4920),
        'Sierra Leone': (8.4606, -11.7799),
        'Somalia': (5.1521, 46.1996),
        'South Africa': (-30.5595, 22.9375),
        'Sudan': (12.8628, 30.2176),
        'Tanzania': (-6.3690, 34.8888),
        'Togo': (8.6195, 0.8248),
        'Tunisia': (33.8869, 9.5375),
        'Uganda': (1.3733, 32.2903),
        'Zambia': (-13.1339, 27.8493),
        'Zimbabwe': (-19.0154, 29.1549),
        
        # Alternative spellings and abbreviations
        'RSA': (-30.5595, 22.9375),  # South Africa
        'SA': (-30.5595, 22.9375),   # South Africa
        'DRC': (-4.0383, 21.7587),   # Democratic Republic of Congo
        'DR Congo': (-4.0383, 21.7587),
        'Congo DR': (-4.0383, 21.7587),
        'Côte d\'Ivoire': (7.5400, -5.5471),  # Ivory Coast
        'CIV': (7.5400, -5.5471),    # Ivory Coast
        'CAR': (6.6111, 20.9394),    # Central African Republic
        'Swaziland': (-26.5225, 31.4659),  # Eswatini (former name)
    }
    
    # Regions
    region_coords = {
        'North Africa': (28.0339, 1.6596),  # Algeria's coordinates as approximation
        'East Africa': (0.0236, 37.9062),   # Kenya's coordinates as approximation
        'West Africa': (9.0820, 8.6753),    # Nigeria's coordinates as approximation
        'Southern Africa': (-30.5595, 22.9375),  # South Africa's coordinates as approximation
        'Central Africa': (6.6111, 20.9394),  # CAR's coordinates as approximation
        'Africa': (9.1450, 40.4897),          # Ethiopia (center of Africa) as approximation
    }
    
    # Major cities
    city_coords = {
        'Abidjan': (5.3600, -4.0083),      # Ivory Coast
        'Accra': (5.6037, -0.1870),        # Ghana
        'Addis Ababa': (9.0320, 38.7460),  # Ethiopia
        'Alexandria': (31.2001, 29.9187),  # Egypt
        'Algiers': (36.7372, 3.0866),      # Algeria
        'Cairo': (30.0444, 31.2357),       # Egypt
        'Cape Town': (-33.9249, 18.4241),  # South Africa
        'Casablanca': (33.5731, -7.5898),  # Morocco
        'Dakar': (14.7167, -17.4677),      # Senegal
        'Dar es Salaam': (-6.7924, 39.2083), # Tanzania
        'Johannesburg': (-26.2041, 28.0473), # South Africa
        'Kampala': (0.3476, 32.5825),      # Uganda
        'Khartoum': (15.5007, 32.5599),    # Sudan
        'Kigali': (-1.9706, 30.1044),      # Rwanda
        'Kinshasa': (-4.4419, 15.2663),    # DRC
        'Lagos': (6.5244, 3.3792),         # Nigeria
        'Nairobi': (-1.2921, 36.8219),     # Kenya
        'Tunis': (36.8065, 10.1815),       # Tunisia
        'Abuja': (9.0765, 7.3986),         # Nigeria
    }
    
    # Combine all dictionaries
    all_coords = {**country_coords, **region_coords, **city_coords}
    
    return all_coords

# Function to match location strings to known locations
def get_fuzzy_location_match(location, coord_dict):
    """
    Try to match a location string with a known location
    """
    # Direct match
    if location in coord_dict:
        return location
    
    # Try case-insensitive match
    location_lower = location.lower()
    for known_loc in coord_dict.keys():
        if known_loc.lower() == location_lower:
            return known_loc
    
    # Try to find the known location as a substring
    for known_loc in coord_dict.keys():
        if known_loc.lower() in location_lower or location_lower in known_loc.lower():
            return known_loc
    
    # Handle common variations
    if "south africa" in location_lower:
        return "South Africa"
    if "nigeria" in location_lower:
        return "Nigeria"
    if "egypt" in location_lower:
        return "Egypt"
    if "tunisia" in location_lower:
        return "Tunisia"
    if "morocco" in location_lower:
        return "Morocco"
    if "algeria" in location_lower:
        return "Algeria"
    if "kenya" in location_lower:
        return "Kenya"
    if "ghana" in location_lower:
        return "Ghana"
    if "ivory" in location_lower or "ivoire" in location_lower:
        return "Ivory Coast"
    if "congo" in location_lower:
        return "Democratic Republic of Congo"
    
    # No match found
    return None

In [59]:
# Prepare location data from all stakeholders
def prepare_location_data(datasets):
    """Extract location data from all stakeholder datasets and normalize to coordinates"""
    
    # Get coordinates dictionary
    coord_dict = get_africa_location_coordinates()
    
    # Get field mappings
    location_fields, id_fields, _ = get_field_mappings()
    
    # Collect all entities with location data
    all_entities = []
    
    # Track entities by location for debugging
    entities_by_location = {}
    
    for entity_type, df in datasets.items():
        # Skip if no location field defined
        loc_field = location_fields.get(entity_type)
        id_field = id_fields.get(entity_type)
        
        if not loc_field or not id_field:
            print(f"Skipping {entity_type} - no location or ID field defined")
            continue
        
        # Process each entity
        for _, row in df.iterrows():
            # Skip if missing location or ID
            if loc_field not in row or pd.isna(row[loc_field]) or id_field not in row or pd.isna(row[id_field]):
                continue
                
            location_str = str(row[loc_field])
            entity_id = row[id_field]
            
            # Try to match location to known coordinates
            matched_loc = get_fuzzy_location_match(location_str, coord_dict)
            
            if matched_loc:
                lat, lon = coord_dict[matched_loc]
                all_entities.append({
                    'entity_type': entity_type,
                    'entity_id': entity_id,
                    'original_location': location_str,
                    'matched_location': matched_loc,
                    'latitude': lat,
                    'longitude': lon
                })
                
                # Add to tracking dictionary
                if matched_loc not in entities_by_location:
                    entities_by_location[matched_loc] = {}
                if entity_type not in entities_by_location[matched_loc]:
                    entities_by_location[matched_loc][entity_type] = 0
                entities_by_location[matched_loc][entity_type] += 1
    
    # Create DataFrame
    location_df = pd.DataFrame(all_entities)
    print(f"Extracted location data for {len(location_df)} entities")
    
    # Print Tunisia specific debugging info
    if 'Tunisia' in entities_by_location:
        print("\nEntities in Tunisia by type:")
        for entity_type, count in entities_by_location['Tunisia'].items():
            print(f"  {entity_type}: {count}")
    
    return location_df

# Create the location_df
location_df = prepare_location_data(datasets)

Skipping managers_staff - no location or ID field defined
Extracted location data for 5371 entities

Entities in Tunisia by type:
  clubs: 30
  suppliers: 22
  fitness_clubs: 6
  traveling_agencies: 3
  sponsors: 1
  sporting_management_agencies: 2
  communication_boxes: 14
  recruiting_agents: 37
  sports_clothing_brands: 43
  players: 308


In [60]:
# Build KNN recommendation model
def build_recommendation_model(location_df, n_neighbors=15):
    """Build a KNN model for location-based recommendations"""
    
    # Ensure coordinates are in float format
    location_df['latitude'] = location_df['latitude'].astype(float)
    location_df['longitude'] = location_df['longitude'].astype(float)
    
    # Extract coordinates
    coords = location_df[['latitude', 'longitude']].values
    
    # Convert to radians for haversine distance
    coords_rad = np.radians(coords)
    
    # Create a mapping from index to location data for quick lookup
    index_map = {i: row for i, row in location_df.reset_index(drop=True).iterrows()}
    
    # Build KNN model
    knn = NearestNeighbors(
        n_neighbors=min(n_neighbors, len(coords_rad)),
        metric='haversine'
    )
    knn.fit(coords_rad)
    
    print(f"Built recommendation model with {len(coords_rad)} entities")
    
    return knn, coords_rad, index_map

# Reset index for KNN model
location_df_reset = location_df.reset_index(drop=True)

# Build model
knn_model, coords_rad, index_map = build_recommendation_model(location_df_reset)

Built recommendation model with 5371 entities


In [61]:
# Utility functions for recommendations
def find_entity_idx(entity_type, entity_id, location_df):
    """Find the index of an entity in the location DataFrame"""
    matches = location_df[
        (location_df['entity_type'] == entity_type) & 
        (location_df['entity_id'] == entity_id)
    ]
    
    if matches.empty:
        return None
    
    # Return the first match's index
    return matches.index[0]

def get_entity_details(entity_type, entity_id, datasets):
    """Get detailed information about an entity"""
    if entity_type not in datasets:
        return None
    
    _, id_fields, _ = get_field_mappings()
    id_field = id_fields.get(entity_type)
    
    if not id_field:
        return None
    
    # Find the entity in the dataset
    entity_data = datasets[entity_type][datasets[entity_type][id_field] == entity_id]
    
    if entity_data.empty:
        return None
    
    return entity_data.iloc[0].to_dict()

def get_entity_name(entity_type, entity_details):
    """Get a display name for an entity based on its type"""
    if not entity_details:
        return f"Unknown {entity_type}"
    
    _, _, name_fields = get_field_mappings()  
    name_field = name_fields.get(entity_type)
    
    if not name_field or name_field not in entity_details:
        return f"Unnamed {entity_type}"
    
    return entity_details[name_field]

# Define friendly names for stakeholder types
stakeholder_type_labels = {
    'clubs': 'Soccer Club',
    'players_agents': 'Player Agent',
    'recruiting_agents': 'Recruiting Agent',
    'service_providers': 'Service Provider',
    'sporting_management_agencies': 'Sports Management Agency',
    'communication_boxes': 'Communication Box',
    'fitness_clubs': 'Fitness Club',
    'suppliers': 'Equipment Supplier',
    'sports_clothing_brands': 'Sports Clothing Brand',
    'traveling_agencies': 'Travel Agency',
    'sponsors': 'Sponsor',
    'managers_staff': 'Manager/Staff',
    'players': 'Player'
}

In [62]:
# Core recommendation functions
def recommend_by_location(entity_type, entity_id, location_df, knn_model, coords_rad, index_map, n_recommendations=300):
    """Recommend entities near a given entity based on location"""
    # Find the entity in our data
    entity_idx = find_entity_idx(entity_type, entity_id, location_df)
    
    if entity_idx is None:
        print(f"Entity {entity_type} with ID {entity_id} not found")
        return None
    
    # Get the entity's coordinates
    entity_coords = coords_rad[entity_idx].reshape(1, -1)
    
    # Find nearest neighbors
    distances, indices = knn_model.kneighbors(entity_coords)
    
    # First index is the entity itself, so skip it
    neighbor_indices = indices[0][1:min(n_recommendations + 20, len(indices[0]))]
    neighbor_distances = distances[0][1:min(n_recommendations + 20, len(distances[0]))]
    
    # Convert distances from radians to kilometers
    neighbor_distances_km = neighbor_distances * 6371  # Earth radius is 6371 km
    
    # Create recommendations DataFrame
    recommendations = []
    for i, idx in enumerate(neighbor_indices):
        neighbor = index_map[idx]
        recommendations.append({
            'entity_type': neighbor['entity_type'],
            'entity_id': neighbor['entity_id'],
            'location': neighbor['matched_location'],
            'distance_km': neighbor_distances_km[i]
        })
    
    return pd.DataFrame(recommendations)

def recommend_other_stakeholders(entity_type, entity_id, location_df, knn_model, coords_rad, index_map, datasets, n_per_type=5):
    """
    Recommend entities from all stakeholder types EXCEPT the entity's own type
    Limited to 5 recommendations per stakeholder type with clear role identification
    """
    # Find the entity to get its location
    entity_matches = location_df[
        (location_df['entity_type'] == entity_type) & 
        (location_df['entity_id'] == entity_id)
    ]
    
    if entity_matches.empty:
        print(f"Entity {entity_type} with ID {entity_id} not found in location data")
        return {}
    
    entity_location = entity_matches.iloc[0]['matched_location']
    
    # Get all recommendations using KNN
    knn_recs = recommend_by_location(
        entity_type, entity_id, location_df, knn_model, coords_rad, index_map, 
        n_recommendations=300  # Get more initially
    )
    
    # Initialize results dictionary
    recommendations_by_type = {}
    
    # Get all unique stakeholder types
    all_stakeholder_types = location_df['entity_type'].unique()
    
    # Process each stakeholder type
    for rec_type in all_stakeholder_types:
        # Skip the source entity type
        if rec_type == entity_type:
            continue
        
        # Get recommendations of this type from KNN results
        type_recs = None
        if knn_recs is not None and not knn_recs.empty:
            type_recs = knn_recs[knn_recs['entity_type'] == rec_type]
        
        # If no recommendations from KNN, try to find entities in the same location
        if type_recs is None or type_recs.empty:
            # Find entities of this type in the same location
            same_loc_entities = location_df[
                (location_df['entity_type'] == rec_type) & 
                (location_df['matched_location'] == entity_location)
            ]
            
            if not same_loc_entities.empty:
                # Create manual recommendations for entities in the same location
                manual_recs = []
                for _, row in same_loc_entities.head(n_per_type).iterrows():
                    manual_recs.append({
                        'entity_type': row['entity_type'],
                        'entity_id': row['entity_id'],
                        'location': row['matched_location'],
                        'distance_km': 0.0  # Same location
                    })
                
                if manual_recs:
                    type_recs = pd.DataFrame(manual_recs)
        
        # If we have recommendations for this type, process them
        if type_recs is not None and not type_recs.empty:
            top_recs = type_recs.head(n_per_type)
            
            # Add detailed information
            detailed_recs = []
            for _, rec in top_recs.iterrows():
                details = get_entity_details(rec['entity_type'], rec['entity_id'], datasets)
                if details:
                    detailed_recs.append({
                        **rec.to_dict(),
                        'details': details
                    })
            
            if detailed_recs:
                recommendations_by_type[rec_type] = detailed_recs
    
    return recommendations_by_type

In [63]:
def test_all_stakeholders(location_df, knn_model, coords_rad, index_map, datasets):
    """
    Test recommendations for one entity from each stakeholder type
    """
    print("\n" + "="*80)
    print("TESTING ALL STAKEHOLDER TYPES")
    print("="*80)
    
    # Get all unique stakeholder types
    all_stakeholder_types = location_df['entity_type'].unique()
    
    for stakeholder_type in all_stakeholder_types:
        print(f"\n{'-'*40}")
        print(f"TESTING: {stakeholder_type.upper()}")
        print(f"{'-'*40}")
        
        # Get a sample entity of this type
        entity_matches = location_df[location_df['entity_type'] == stakeholder_type]
        
        if entity_matches.empty:
            print(f"No entities found for {stakeholder_type}")
            continue
            
        # Get first entity to test (or you could use sample() for random)
        sample_entity = entity_matches.iloc[0]
        entity_id = sample_entity['entity_id']
        
        # Get entity details
        entity_details = get_entity_details(stakeholder_type, entity_id, datasets)
        if not entity_details:
            print(f"Entity details not found in dataset")
            continue
            
        entity_name = get_entity_name(stakeholder_type, entity_details)
        print(f"Testing entity: {entity_name}")
        print(f"Location: {sample_entity['matched_location']}")
        
        # Get recommendations
        print("Generating recommendations...")
        recommendations = recommend_other_stakeholders(
            stakeholder_type, entity_id, location_df, 
            knn_model, coords_rad, index_map, datasets, 
            n_per_type=5  # Show 5 per type
        )
        
        if not recommendations:
            print("No recommendations found.")
            continue
            
        # Print recommendations with clear role identification
        print(f"Top recommendations from {len(recommendations)} stakeholder types:")
        
        for rec_type, recs in recommendations.items():
            # Get friendly name for the stakeholder type
            type_label = stakeholder_type_labels.get(rec_type, rec_type.replace('_', ' ').title())
            print(f"\n{type_label.upper()} RECOMMENDATIONS:")
            
            for i, rec in enumerate(recs, 1):
                rec_name = get_entity_name(rec_type, rec['details'])
                
                print(f"  {i}. {rec_name} ({type_label})")
                print(f"     Location: {rec['location']} ({rec['distance_km']:.1f} km)")
                
                # Add type-specific details for all stakeholder types
                if rec_type == 'players':
                    print(f"     Position: {rec['details'].get('position', 'N/A')}")
                    print(f"     Age: {rec['details'].get('age', 'N/A')}")
                
                elif rec_type == 'clubs':
                    print(f"     League: {rec['details'].get('league', 'N/A')}")
                    print(f"     Stadium: {rec['details'].get('stadium', 'N/A')}")
                
                elif rec_type == 'players_agents':
                    print(f"     Experience: {rec['details'].get('years_experience', 'N/A')} years")
                    print(f"     Success Rate: {rec['details'].get('success_rate', 'N/A')}%")
                
                elif rec_type == 'sponsors':
                    print(f"     Industry: {rec['details'].get('industry', 'N/A')}")
                    print(f"     Contract Value: {rec['details'].get('contract_value', 'N/A')}")
                
                elif rec_type == 'fitness_clubs':
                    print(f"     Services: {rec['details'].get('services_offered', 'N/A')}")
                    print(f"     Certified Trainers: {rec['details'].get('certified_trainers', 'N/A')}")
                
                elif rec_type == 'suppliers':
                    print(f"     Product Type: {rec['details'].get('product_type', 'N/A')}")
                    print(f"     Price: {rec['details'].get('price', 'N/A')}")
                
                elif rec_type == 'traveling_agencies':
                    print(f"     Services: {rec['details'].get('services_offered', 'N/A')}")
                
                elif rec_type == 'recruiting_agents':
                    print(f"     Experience: {rec['details'].get('experience_years', 'N/A')} years")
                    print(f"     Positions: {rec['details'].get('preferred_positions', 'N/A')}")
                
                elif rec_type == 'service_providers':
                    print(f"     Service Type: {rec['details'].get('service_type', 'N/A')}")
                
                elif rec_type == 'sporting_management_agencies':
                    print(f"     Services: {rec['details'].get('services_offered', 'N/A')}")
                
                elif rec_type == 'communication_boxes':
                    print(f"     Services: {rec['details'].get('services_offered', 'N/A')}")
                
                elif rec_type == 'sports_clothing_brands':
                    print(f"     Products: {rec['details'].get('product_types', 'N/A')}")
                
                elif rec_type == 'managers_staff':
                    print(f"     Function: {rec['details'].get('function', 'N/A')}")
                    print(f"     Experience: {rec['details'].get('years_experience', 'N/A')} years")

In [ ]:
def quick_test_specific_entities(location_df, knn_model, coords_rad, index_map, datasets):
    """
    Test recommendations for specific pre-defined entities
    """
    print("\n" + "="*80)
    print("TESTING SPECIFIC ENTITIES")
    print("="*80)
    
    # Define specific entities to test
    test_entities = [
        {'type': 'players', 'id': 7},

        # Add other entities to test as needed
    ]
    
    for entity in test_entities:
        entity_type = entity['type']
        entity_id = entity['id']
        
        print(f"\n{'-'*40}")
        print(f"TESTING: {entity_type.upper()} with ID {entity_id}")
        print(f"{'-'*40}")
        
        # Find the entity in our data
        entity_idx = find_entity_idx(entity_type, entity_id, location_df)
        
        if entity_idx is None:
            print(f"No entity found with type={entity_type}, id={entity_id}")
            continue
            
        # Get entity details
        entity_matches = location_df[
            (location_df['entity_type'] == entity_type) & 
            (location_df['entity_id'] == entity_id)
        ]
        
        if entity_matches.empty:
            print(f"No entity found in location data")
            continue
            
        entity_row = entity_matches.iloc[0]
        
        # Get entity details
        entity_details = get_entity_details(entity_type, entity_id, datasets)
        if not entity_details:
            print(f"Entity details not found in dataset")
            continue
            
        entity_name = get_entity_name(entity_type, entity_details)
        print(f"Found entity: {entity_name}")
        print(f"Location: {entity_row['matched_location']}")
        
        # Print Tunisia-specific entities for debugging
        print(f"Analyzing entities in {entity_row['matched_location']}...")
        tunisia_entities = location_df[location_df['matched_location'] == entity_row['matched_location']]
        type_counts = tunisia_entities['entity_type'].value_counts()
        print("Stakeholder types in this location:")
        for type_name, count in type_counts.items():
            print(f"  - {type_name}: {count} entities")
        
        # Get recommendations
        print("Generating recommendations...")
        recommendations = recommend_other_stakeholders(
            entity_type, entity_id, location_df, 
            knn_model, coords_rad, index_map, datasets, 
            n_per_type=5
        )
        
        if not recommendations:
            print("No recommendations found.")
            continue
            
        # Print recommendations
        print(f"Top recommendations from {len(recommendations)} stakeholder types:")
        
        for rec_type, recs in recommendations.items():
            # Get friendly name for the stakeholder type
            type_label = stakeholder_type_labels.get(rec_type, rec_type.replace('_', ' ').title())
            print(f"\n{type_label.upper()} RECOMMENDATIONS:")
            
            for i, rec in enumerate(recs, 1):
                rec_name = get_entity_name(rec_type, rec['details'])
                
                print(f"  {i}. {rec_name} ({type_label})")
                print(f"     Location: {rec['location']} ({rec['distance_km']:.1f} km)")
                
                # Add type-specific details based on stakeholder type
                if rec_type == 'clubs':
                    print(f"     League: {rec['details'].get('league', 'N/A')}")
                elif rec_type == 'players':
                    print(f"     Position: {rec['details'].get('position', 'N/A')}")
                    print(f"     Age: {rec['details'].get('age', 'N/A')}")
                elif rec_type == 'players_agents':
                    print(f"     Experience: {rec['details'].get('years_experience', 'N/A')} years")
                elif rec_type == 'fitness_clubs':
                    print(f"     Services: {rec['details'].get('services_offered', 'N/A')}")
                elif rec_type == 'sponsors':
                    print(f"     Industry: {rec['details'].get('industry', 'N/A')}")
                elif rec_type == 'suppliers':
                    print(f"     Product Type: {rec['details'].get('product_type', 'N/A')}")
                elif rec_type == 'traveling_agencies':
                    print(f"     Services: {rec['details'].get('services_offered', 'N/A')}")

In [67]:
# Run the non-interactive test with pre-defined entities
quick_test_specific_entities(location_df_reset, knn_model, coords_rad, index_map, datasets)


TESTING SPECIFIC ENTITIES

----------------------------------------
TESTING: PLAYERS with ID 7
----------------------------------------
Found entity: Amenallah Memmiche
Location: Tunisia
Analyzing entities in Tunisia...
Stakeholder types in this location:
  - players: 308 entities
  - sports_clothing_brands: 43 entities
  - recruiting_agents: 37 entities
  - clubs: 30 entities
  - suppliers: 22 entities
  - communication_boxes: 14 entities
  - fitness_clubs: 6 entities
  - traveling_agencies: 3 entities
  - sporting_management_agencies: 2 entities
  - sponsors: 1 entities
Generating recommendations...
Top recommendations from 9 stakeholder types:

SOCCER CLUB RECOMMENDATIONS:
  1. Etoile du Sahel (Soccer Club)
     Location: Tunisia (0.0 km)
     League: Tunisian Ligue Professionnelle 1
  2. CA Bizertin (Soccer Club)
     Location: Tunisia (0.0 km)
     League: Tunisian Ligue 2
  3. Esperance de Tunis (Soccer Club)
     Location: Tunisia (0.0 km)
     League: Tunisian Ligue Profession